# A

In [1]:
"""
End-to-end, *pure supervised* training for a JSON diagram-spec generator
with permutation-invariant regression heads (Hungarian matching), built on
Hugging Face causal LMs.
"""

import os, gc, torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
gc.collect(); torch.cuda.empty_cache()

import os, math, json, random
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

try:
    from scipy.optimize import linear_sum_assignment
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

# 0) Speed/Memory knobs (A100+ friendly)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# 1) Taxonomy & parameterization

TYPE_ID = {
    "block": 0,
    "incline": 1,
    "horizontal": 2,
    "force_arrow": 3,
    "axis_origin": 4,
}

ID_TYPE = {v: k for k, v in TYPE_ID.items()}
MAX_P = 8  # max parameter slots per object (unused are masked off)

SPECIAL_TOKENS = {"additional_special_tokens": ["<OBJ>", "</OBJS>"]}


def get_d_model(cfg):
    if getattr(cfg, "hidden_size", None) is not None:
        return cfg.hidden_size
    if getattr(cfg, "n_embd", None) is not None:
        return cfg.n_embd
    raise ValueError("Could not determine d_model from config.")


def angle_deg_to_rad_norm(deg: float) -> float:
    r = math.radians(deg) % (2 * math.pi)
    return r / (2 * math.pi)


def norm_xy(x: float, y: float, xlim: Tuple[float, float], ylim: Tuple[float, float]) -> Tuple[float, float]:
    xmin, xmax = xlim; ymin, ymax = ylim
    nx = (x - xmin) / (xmax - xmin + 1e-6)
    ny = (y - ymin) / (ymax - ymin + 1e-6)
    return float(nx), float(ny)


def build_extents(data: Dict[str, Any]):
    # Fixed window for demo; swap to scene bounds if you want
    return (-3.0, 3.0), (-3.0, 3.0)


def flatten_example(data: Dict[str, Any]):
    (xlim, ylim) = build_extents(data)
    types: List[int] = []
    params: List[List[float]] = []
    masks: List[List[int]] = []

    # bodies
    for b in data.get("bodies", []):
        if b.get("type") == "block":
            cx, cy = norm_xy(b["position"]["x"], b["position"]["y"], xlim, ylim)
            w = b["size"]["width"] / (xlim[1] - xlim[0] + 1e-6)
            h = b["size"]["height"] / (ylim[1] - ylim[0] + 1e-6)
            theta = angle_deg_to_rad_norm(b.get("rotation_deg", 0.0))
            p = [cx, cy, w, h, theta]; m = [1, 1, 1, 1, 1]
            types.append(TYPE_ID["block"]); params.append(p + [0.0] * (MAX_P - len(p))); masks.append(m + [0] * (MAX_P - len(m)))

    # surfaces
    for s in data.get("surfaces", []):
        if s["type"] == "incline":
            ang = angle_deg_to_rad_norm(s["angle_deg"])
            tx, ty = norm_xy(s["through"]["x"], s["through"]["y"], xlim, ylim)
            length = min(1.0, s.get("length", 6.0) / 10.0)
            p = [ang, tx, ty, length]; m = [1, 1, 1, 1]
            types.append(TYPE_ID["incline"]); params.append(p + [0.0] * (MAX_P - len(p))); masks.append(m + [0] * (MAX_P - len(m)))
        elif s["type"] in ("horizontal", "ground"):
            ny = norm_xy(0.0, s.get("y", 0.0), xlim, ylim)[1]
            ncx = norm_xy(s.get("center", {}).get("x", 0.0), 0.0, xlim, ylim)[0]
            length = min(1.0, s.get("length", 6.0) / 10.0)
            p = [ny, ncx, length]; m = [1, 1, 1]
            types.append(TYPE_ID["horizontal"]); params.append(p + [0.0] * (MAX_P - len(p))); masks.append(m + [0] * (MAX_P - len(m)))

    # forces -> single geometric type force_arrow
    for f in data.get("forces", []):
        a = f.get("arrow", {})
        sx, sy = norm_xy(a.get("start", {}).get("x", 0.0), a.get("start", {}).get("y", 0.0), xlim, ylim)
        ang = angle_deg_to_rad_norm(a.get("angle_deg", 0.0))
        raw_len = a.get("length", 1.0) / (xlim[1] - xlim[0] + 1e-6)
        length = max(0.0, min(1.0, raw_len))
        p = [sx, sy, ang, length]; m = [1, 1, 1, 1]
        types.append(TYPE_ID["force_arrow"]); params.append(p + [0.0] * (MAX_P - len(p))); masks.append(m + [0] * (MAX_P - len(m)))

    # labels -> axis_origin
    lbl = data.get("labels", {})
    if lbl.get("show_axes", False):
        ox, oy = norm_xy(lbl.get("origin", {}).get("x", 0.0), lbl.get("origin", {}).get("y", 0.0), xlim, ylim)
        p = [ox, oy]; m = [1, 1]
        types.append(TYPE_ID["axis_origin"]); params.append(p + [0.0] * (MAX_P - len(p))); masks.append(m + [0] * (MAX_P - len(m)))

    if len(types) == 0:
        return (torch.zeros(0, dtype=torch.long), torch.zeros(0, MAX_P), torch.zeros(0, MAX_P, dtype=torch.long))

    return (
        torch.tensor(types, dtype=torch.long),
        torch.tensor(params, dtype=torch.float32),
        torch.tensor(masks, dtype=torch.long),
    )


# 2) ObjHead (exist/cls/reg)

class ObjHead(nn.Module):
    def __init__(self, d_model: int, num_types: int, max_params: int = MAX_P):
        super().__init__()
        self.exist = nn.Linear(d_model, 1)
        self.cls = nn.Linear(d_model, num_types)
        self.reg = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, max_params))

    def forward(self, h: torch.Tensor):  # h: [K, D]
        exist_logit = self.exist(h).squeeze(-1)
        cls_logit = self.cls(h)
        reg_vec = self.reg(h)
        return exist_logit, cls_logit, reg_vec


# 3) Matching cost & supervised set loss

ANGLE_IDXS_BY_TYPE = {
    TYPE_ID["block"]: [4],       # theta
    TYPE_ID["incline"]: [0],     # angle
    TYPE_ID["force_arrow"]: [2], # angle
}


def build_cost(cls_logit, reg_vec, gold_types, gold_regs, gold_masks, w_type=1.0, w_l1=1.0, w_angle=0.5):
    K, T = cls_logit.shape
    M, P = gold_regs.shape
    type_cost = F.cross_entropy(
        cls_logit.unsqueeze(1).expand(K, M, T).reshape(K * M, T),
        gold_types.unsqueeze(0).expand(K, M).reshape(K * M),
        reduction="none",
    ).view(K, M)
    pred = reg_vec.unsqueeze(1).expand(K, M, P)
    gold = gold_regs.unsqueeze(0).expand(K, M, P)
    mask = gold_masks.unsqueeze(0).expand(K, M, P).float()
    l1 = (mask * (pred - gold).abs()).sum(-1) / (mask.sum(-1) + 1e-6)
    angle_pen = torch.zeros_like(l1)
    for t, idxs in ANGLE_IDXS_BY_TYPE.items():
        m_t = (gold_types[None, :] == t)
        if not m_t.any() or len(idxs) == 0:
            continue
        pred_a = pred[:, :, idxs]
        gold_a = gold[:, :, idxs]
        d = 2 * math.pi * (pred_a - gold_a)
        angle_pen = angle_pen + m_t.float() * (1 - torch.cos(d)).mean(-1)
    return w_type * type_cost + w_l1 * l1 + w_angle * angle_pen


def hungarian_match(cls_logit, reg_vec, gold_types, gold_regs, gold_masks):
    C = build_cost(cls_logit, reg_vec, gold_types, gold_regs, gold_masks)
    if SCIPY_OK:
        r, c = linear_sum_assignment(C.detach().cpu().numpy())
        return torch.as_tensor(r, device=cls_logit.device), torch.as_tensor(c, device=cls_logit.device)
    # Greedy fallback when scipy missing
    K, M = C.shape
    r_idx, c_idx, used_r, used_c = [], [], set(), set()
    for f in C.detach().cpu().flatten().argsort().tolist():
        r = f // M; c = f % M
        if r not in used_r and c not in used_c:
            r_idx.append(r); c_idx.append(c); used_r.add(r); used_c.add(c)
        if len(used_c) == min(K, M):
            break
    return torch.tensor(r_idx, device=cls_logit.device), torch.tensor(c_idx, device=cls_logit.device)


def supervised_set_loss(slot_states, obj_head, gold_types, gold_regs, gold_masks, lambda_exist=1.0, lambda_cls=1.0, lambda_reg=1.0):
    exist_logit, cls_logit, reg_vec = obj_head(slot_states)
    K = exist_logit.shape[0]; device = exist_logit.device
    if gold_types.numel() == 0:
        target_exist = torch.zeros(K, device=device)
        exist_loss = F.binary_cross_entropy_with_logits(exist_logit, target_exist)
        return lambda_exist * exist_loss, {"exist": exist_loss.item(), "cls": 0.0, "reg": 0.0}
    rows, cols = hungarian_match(cls_logit, reg_vec, gold_types, gold_regs, gold_masks)
    matched = torch.zeros(K, dtype=torch.bool, device=device); matched[rows] = True
    target_exist = torch.zeros(K, device=device); target_exist[matched] = 1.0
    exist_loss = F.binary_cross_entropy_with_logits(exist_logit, target_exist)
    cls_loss = F.cross_entropy(cls_logit[rows], gold_types[cols])
    pred = reg_vec[rows]; gold = gold_regs[cols]; mask = gold_masks[cols].float()
    reg_l1 = (mask * F.smooth_l1_loss(pred, gold, reduction="none")).sum() / (mask.sum() + 1e-6)
    total = lambda_exist * exist_loss + lambda_cls * cls_loss + lambda_reg * reg_l1
    return total, {"exist": exist_loss.item(), "cls": cls_loss.item(), "reg": reg_l1.item()}


# 4) Optional overlap penalty for boxes

def boxes_to_corners(cx, cy, w, h):
    x1 = cx - 0.5 * w; y1 = cy - 0.5 * h
    x2 = cx + 0.5 * w; y2 = cy + 0.5 * h
    return x1, y1, x2, y2


def soft_iou_loss(pred_boxes: torch.Tensor, alpha: float = 0.0) -> torch.Tensor:
    if pred_boxes.numel() == 0:
        return torch.tensor(0.0, device=pred_boxes.device)
    x1, y1, x2, y2 = boxes_to_corners(pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3])
    ix1 = torch.maximum(x1[:, None], x1[None, :])
    iy1 = torch.maximum(y1[:, None], y1[None, :])
    ix2 = torch.minimum(x2[:, None], x2[None, :])
    iy2 = torch.minimum(y2[:, None], y2[None, :])
    iw = torch.clamp(ix2 - ix1, min=0)
    ih = torch.clamp(iy2 - iy1, min=0)
    inter = iw * ih
    area = (x2 - x1) * (y2 - y1)
    union = area[:, None] + area[None, :] - inter + 1e-6
    iou = inter / union
    mask = torch.triu(torch.ones_like(iou), diagonal=1).bool()
    return torch.relu(iou[mask] - alpha).mean() if mask.any() else torch.tensor(0.0, device=pred_boxes.device)


# 5) Dataset + Data collator

TOY_PROBLEM = "A block of mass m rests on a 30° incline. Draw the FBD with W, N, and friction f."
TOY_JSON = {
    "bodies": [
        {"id": "block1", "type": "block", "position": {"x": 0, "y": 0}, "size": {"width": 0.6, "height": 0.4}, "rotation_deg": 30, "label": ""}
    ],
    "surfaces": [
        {"id": "incline", "type": "incline", "angle_deg": 30, "through": {"x": -0.4, "y": -0.5}, "length": 5.0}
    ],
    "forces": [
        {"id": "W", "type": "weight", "on": "block1",
         "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
         "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
        {"id": "N", "type": "normal", "on": "block1",
         "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 120, "length": 0.9},
         "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
        {"id": "f", "type": "friction", "on": "block1",
         "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 30, "length": 0.8},
         "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
    ],
    "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
}

problems = ["A block of mass m rests on a 30° incline. Draw a free body diagram with W, N, and friction f.",
            "A block of mass m rests on a 45° incline. Draw a free body diagram with W, N, and friction f."]

jsons = [
    {
        "bodies": [
            {"id": "block1", "type": "block", "position": {"x": 0, "y": 0}, "size": {"width": 0.6, "height": 0.4}, "rotation_deg": 30, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline", "angle_deg": 30, "through": {"x": -0.4, "y": -0.5}, "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
            "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 120, "length": 0.9},
            "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 30, "length": 0.8},
            "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    },
    {
        "bodies": [
            {"id": "block1", "type": "block", "position": {"x": 0, "y": 0}, "size": {"width": 0.6, "height": 0.4}, "rotation_deg": 45, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline", "angle_deg": 45, "through": {"x": -0.4, "y": -0.7}, "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
            "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 135, "length": 0.9},
            "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
            "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 45, "length": 0.8},
            "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    }
]


def inject_obj_tokens(pretty_json: str) -> str:
    s = pretty_json
    s = s.replace("\n    {\"id\":", "\n    <OBJ> {\"id\":")
    s = s.replace("\n        {\"id\":", "\n        <OBJ> {\"id\":")
    return s + "\n</OBJS>"


class DiagramDataset(Dataset):
    def __init__(self, num_samples: int = 8):
        self.samples = []
        for _ in range(num_samples):
            random_idx = random.randint(0, len(problems) - 1)
            problem = problems[random_idx]
            j = json.loads(json.dumps(jsons[random_idx]))
            j["bodies"][0]["position"]["x"] += random.uniform(-0.05, 0.05)
            j["bodies"][0]["position"]["y"] += random.uniform(-0.05, 0.05)
            j["forces"][0]["arrow"]["length"] += random.uniform(-0.1, 0.1)
            self.samples.append({"problem": problem, "json": j})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        problem = item["problem"]
        j = item["json"]
        target_json = json.dumps(j, ensure_ascii=False, indent=2)
        target_with_markers = inject_obj_tokens(target_json)
        g_types, g_regs, g_masks = flatten_example(j)
        return {
            "problem": problem,
            "target_text": target_with_markers,
            "gold_types": g_types,
            "gold_regs": g_regs,
            "gold_masks": g_masks,
        }

"""
@dataclass
class DataCollator:
    tokenizer: PreTrainedTokenizerBase
    max_len: int = 256

    def __call__(self, batch):
        # Build prompt and target
        prompts  = [b["problem"] for b in batch]
        targets  = [b["target_text"] for b in batch]

        # You can wrap the prompt with a simple instruction to bias format
        prefixed_prompts = [
            "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
            + p + "\n\nJSON:\n"
            for p in prompts
        ]

        enc_prompt = self.tokenizer(
            prefixed_prompts, padding=True, truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        enc_target = self.tokenizer(
            targets, padding=True, truncation=True, max_length=self.max_len, return_tensors="pt"
        )

        # concat: [prompt | target]
        input_ids  = []
        attn_mask  = []
        labels     = []
        obj_positions = []

        obj_tok_id = self.tokenizer.convert_tokens_to_ids("<OBJ>")
        pad_id = self.tokenizer.pad_token_id

        for i in range(enc_prompt.input_ids.size(0)):
            p_ids = enc_prompt.input_ids[i]
            p_att = enc_prompt.attention_mask[i]
            t_ids = enc_target.input_ids[i]
            t_att = enc_target.attention_mask[i]

            # concat and pad/truncate to max_len
            ids  = torch.cat([p_ids, t_ids], dim=0)[:self.max_len]
            att  = torch.cat([p_att, t_att], dim=0)[:self.max_len]

            # labels: ignore prompt tokens
            lab  = torch.full_like(ids, fill_value=-100)
            t_len = min(t_att.sum().item(), self.max_len - int(p_att.sum().item()))
            # place target labels right after the prompt span
            start = int(p_att.sum().item())
            end   = start + int(t_len)
            lab[start:end] = ids[start:end]

            # find <OBJ> positions only inside the target span
            obj_pos = (ids[start:end] == obj_tok_id).nonzero(as_tuple=False).flatten().add(start).tolist()

            # right-pad if truncation shortened things (HF already did padding above)
            input_ids.append(ids)
            attn_mask.append(att)
            labels.append(lab)
            obj_positions.append(obj_pos)

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn_mask),
            "labels": torch.stack(labels),
            "obj_positions": obj_positions,
            "gold_types": [b["gold_types"] for b in batch],
            "gold_regs":  [b["gold_regs"]  for b in batch],
            "gold_masks": [b["gold_masks"] for b in batch],
        }
"""
from dataclasses import dataclass
from typing import List, Dict, Any
import torch
from transformers import PreTrainedTokenizerBase

@dataclass
class DataCollator:
    tokenizer: PreTrainedTokenizerBase
    max_len: int = 1024

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        # 1) Build prompts and targets
        prompts = [b["problem"] for b in batch]
        targets = [b["target_text"] for b in batch]

        # Instruction prefix (must match what you use at inference)
        prefixed_prompts = [
            "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\n"
            "Problem:\n" + p + "\n\nJSON:\n"
            for p in prompts
        ]

        enc_p = self.tokenizer(
            prefixed_prompts,
            padding=False, truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        enc_t = self.tokenizer(
            targets,
            padding=False, truncation=True, max_length=self.max_len, return_tensors="pt"
        )

        obj_tok_id = self.tokenizer.convert_tokens_to_ids("<OBJ>")
        pad_id = self.tokenizer.pad_token_id

        input_ids_list: List[torch.Tensor] = []
        labels_list: List[torch.Tensor] = []
        attn_list: List[torch.Tensor] = []
        obj_positions: List[List[int]] = []

        B = enc_p.input_ids.size(0)
        for i in range(B):
            p_ids = enc_p.input_ids[i]
            p_att = enc_p.attention_mask[i]
            t_ids = enc_t.input_ids[i]
            t_att = enc_t.attention_mask[i]

            # concat then hard-truncate to max_len
            ids = torch.cat([p_ids, t_ids], dim=0)
            att = torch.cat([p_att, t_att], dim=0)
            if ids.size(0) > self.max_len:
                ids = ids[: self.max_len]
                att = att[: self.max_len]

            # compute spans (prompt length = number of non-pad tokens in p_att, capped by max_len)
            plen = min(int(p_att.sum().item()), self.max_len)
            # available room for target tokens
            avail_t = self.max_len - plen
            tlen = min(int(t_att.sum().item()), max(0, avail_t))

            # 2) Build labels: mask prompt, supervise target
            lab = torch.full_like(ids, fill_value=-100)
            start = plen
            end = plen + tlen
            if end > start:
                lab[start:end] = ids[start:end]
                supervised = int((lab != -100).sum().item())
                if supervised == 0:
                    print("[WARN] zero supervised tokens: prompt ate the whole sequence. Lower max prompt length or raise max_len.")

            # 3) Right-pad to max_len if needed
            if ids.size(0) < self.max_len:
                pad_len = self.max_len - ids.size(0)
                ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=ids.dtype)], dim=0)
                lab = torch.cat([lab, torch.full((pad_len,), -100, dtype=lab.dtype)], dim=0)
                att = torch.cat([att, torch.zeros(pad_len, dtype=att.dtype)], dim=0)

            # 4) Find <OBJ> positions only within the (supervised) target span
            # Note: end may be 0 if target was fully truncated; handle safely.
            obj_pos = []
            if end > start:
                target_slice = ids[start:end]
                rel_idx = (target_slice == obj_tok_id).nonzero(as_tuple=False).flatten()
                obj_pos = (rel_idx + start).tolist()

            input_ids_list.append(ids)
            labels_list.append(lab)
            attn_list.append(att)
            obj_positions.append(obj_pos)

        input_ids = torch.stack(input_ids_list)
        attention_mask = torch.stack(attn_list).long()
        labels = torch.stack(labels_list)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "obj_positions": obj_positions,
            # pass-through gold tensors for your set loss
            "gold_types": [b["gold_types"] for b in batch],
            "gold_regs":  [b["gold_regs"]  for b in batch],
            "gold_masks": [b["gold_masks"] for b in batch],
        }




# 6) Model wrapper for HF Trainer
"""
class DiagramWrapper(nn.Module):
    def __init__(self, backbone, obj_head, pad_token_id,
                 lambda_text=1.0, lambda_exist=1.0, lambda_cls=1.0, lambda_reg=1.0, lambda_overlap=0.0):
        super().__init__()
        self.backbone = backbone
        self.obj_head = obj_head
        self.pad_token_id = pad_token_id
        self.lambda_text = lambda_text
        self.lambda_exist = lambda_exist
        self.lambda_cls = lambda_cls
        self.lambda_reg = lambda_reg
        self.lambda_overlap = lambda_overlap

    def forward(self, **batch):
        input_ids      = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels         = batch["labels"]
        obj_positions  = batch["obj_positions"]
        gold_types     = batch["gold_types"]
        gold_regs      = batch["gold_regs"]
        gold_masks     = batch["gold_masks"]

        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,             # CE only on target span (prompt masked as -100)
            output_hidden_states=True,
            return_dict=True,
        )
        text_loss = out.loss
        hidden = out.hidden_states[-1]  # [B, L, D]

        # extract slot states at <OBJ> positions (these are in the target region)
        exist_losses, cls_losses, reg_losses = [], [], []
        for b in range(hidden.size(0)):
            pos = obj_positions[b]
            if not pos:
                continue
            Hslots = hidden[b, torch.as_tensor(pos, device=hidden.device)]
            g_types = gold_types[b].to(hidden.device)
            g_regs  = gold_regs[b].to(hidden.device)
            g_masks = gold_masks[b].to(hidden.device)
            set_loss, logs = supervised_set_loss(
                Hslots, self.obj_head, g_types, g_regs, g_masks
            )
            exist_losses.append(torch.tensor(logs["exist"], device=hidden.device))
            cls_losses.append(torch.tensor(logs["cls"], device=hidden.device))
            reg_losses.append(torch.tensor(logs["reg"], device=hidden.device))

        exist_loss = torch.stack(exist_losses).mean() if exist_losses else torch.tensor(0., device=hidden.device)
        cls_loss   = torch.stack(cls_losses).mean()   if cls_losses   else torch.tensor(0., device=hidden.device)
        reg_loss   = torch.stack(reg_losses).mean()   if reg_losses   else torch.tensor(0., device=hidden.device)

        total = text_loss + exist_loss + cls_loss + reg_loss
        return {"loss": total}
"""
class DiagramWrapper(nn.Module):
    def __init__(self, backbone, obj_head, pad_token_id,
                 lambda_text=1.0, lambda_exist=1.0, lambda_cls=1.0, lambda_reg=1.0):
        super().__init__()
        self.backbone = backbone
        self.obj_head = obj_head
        self.pad_token_id = pad_token_id
        self.lambda_text = lambda_text
        self.lambda_exist = lambda_exist
        self.lambda_cls = lambda_cls
        self.lambda_reg = lambda_reg

    def forward(self, **batch):
        input_ids      = batch["input_ids"]           # [B, L]
        attention_mask = batch["attention_mask"]      # [B, L]
        labels         = batch["labels"]              # [B, L], prompt masked to -100
        obj_positions  = batch["obj_positions"]       # list(list(int))
        gold_types     = batch["gold_types"]
        gold_regs      = batch["gold_regs"]
        gold_masks     = batch["gold_masks"]

        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,                 # HF computes token CE using -100 mask
            output_hidden_states=True,
            return_dict=True,
        )
        text_loss = out.loss
        hidden = out.hidden_states[-1]     # [B, L, D]

        exist_losses, cls_losses, reg_losses = [], [], []
        for b in range(hidden.size(0)):
            pos = obj_positions[b]
            if not pos:
                continue
            Hslots  = hidden[b, torch.as_tensor(pos, device=hidden.device)]  # [K, D]
            g_types = gold_types[b].to(hidden.device)
            g_regs  = gold_regs[b].to(hidden.device)
            g_masks = gold_masks[b].to(hidden.device)
            set_loss, logs = supervised_set_loss(
                Hslots, self.obj_head, g_types, g_regs, g_masks,
                lambda_exist=self.lambda_exist, lambda_cls=self.lambda_cls, lambda_reg=self.lambda_reg,
            )
            exist_losses.append(torch.tensor(logs["exist"], device=hidden.device))
            cls_losses.append(torch.tensor(logs["cls"],   device=hidden.device))
            reg_losses.append(torch.tensor(logs["reg"],   device=hidden.device))

        exist_loss = (torch.stack(exist_losses).mean()
                      if exist_losses else torch.tensor(0.0, device=hidden.device))
        cls_loss   = (torch.stack(cls_losses).mean()
                      if cls_losses else torch.tensor(0.0, device=hidden.device))
        reg_loss   = (torch.stack(reg_losses).mean()
                      if reg_losses else torch.tensor(0.0, device=hidden.device))

        loss = self.lambda_text * text_loss + exist_loss + cls_loss + reg_loss
        return {"loss": loss}



# 7) Tiny training

def main():
    # Pick a small default that fits everywhere; switch to Qwen/Mistral below
    backbone_name = "Qwen/Qwen2.5-Coder-7B-Instruct" #"TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # ==== TOKENIZER/MODEL LOAD ====
    tokenizer = AutoTokenizer.from_pretrained(backbone_name, trust_remote_code=True)
    added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- Option A: standard fp/bf16 full model (use for small backbones) ---
    backbone = AutoModelForCausalLM.from_pretrained(
        backbone_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else None,
        device_map=None,  # single-GPU training
    )
    backbone.config.use_cache = False
    backbone.gradient_checkpointing_enable()
    if added > 0:
        backbone.resize_token_embeddings(len(tokenizer))
    if getattr(backbone.config, "pad_token_id", None) is None:
        backbone.config.pad_token_id = tokenizer.pad_token_id

    # --- Option B: 4-bit + LoRA for big models (Qwen/Mistral/Llama) ---
    # from transformers import BitsAndBytesConfig
    # from peft import LoraConfig, get_peft_model
    # bnb_config = BitsAndBytesConfig(
    #     load_in_4bit=True,
    #     bnb_4bit_compute_dtype=torch.bfloat16,
    #     bnb_4bit_use_double_quant=True,
    #     bnb_4bit_quant_type="nf4",
    # )
    # backbone = AutoModelForCausalLM.from_pretrained(
    #     "Qwen/Qwen2.5-Coder-7B-Instruct",
    #     trust_remote_code=True,
    #     quantization_config=bnb_config,
    # )
    # backbone.config.use_cache = False
    # backbone.gradient_checkpointing_enable()
    # if added > 0:
    #     backbone.resize_token_embeddings(len(tokenizer))
    # if getattr(backbone.config, "pad_token_id", None) is None:
    #     backbone.config.pad_token_id = tokenizer.pad_token_id
    # lora_cfg = LoraConfig(
    #     r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    #     target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    # )
    # backbone = get_peft_model(backbone, lora_cfg)

    obj_head = ObjHead(d_model=get_d_model(backbone.config), num_types=len(TYPE_ID), max_params=MAX_P)
    model = DiagramWrapper(backbone, obj_head, pad_token_id=tokenizer.pad_token_id)

    # ==== DATA ====
    ds = DiagramDataset(num_samples=8)
    collator = DataCollator(tokenizer=tokenizer, max_len=256)

    # ==== TRAINER ====
    args = TrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        num_train_epochs=50,
        logging_steps=5,
        save_steps=0,
        report_to=[],
        remove_unused_columns=False,
        fp16=False,
        bf16=torch.cuda.is_available(),
    )

    class HFWrapper(nn.Module):
        def __init__(self, core: DiagramWrapper):
            super().__init__()
            self.core = core
        def forward(self, **batch):
            out = self.core(**batch)
            return out["loss"]

    trainer = Trainer(
        model=model, #HFWrapper(model),
        args=args,
        train_dataset=ds,
        data_collator=collator,
    )

    trainer.train()
    print("Training complete. Supervised JSON+regression model is trained on toy data.")

    # === Generate & save predictions on toy dataset ===
    os.makedirs("./out", exist_ok=True)
    pred_path = "./out/predictions.jsonl"

    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    EOS    = tokenizer.eos_token_id
    PAD    = tokenizer.pad_token_id

    with open(pred_path, "w", encoding="utf-8") as f:
        for ex in ds:
            # Use the same prompt format you trained on
            prompt = (
                "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\n"
                "Problem:\n" + ex["problem"] + "\n\nJSON:\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.backbone.device)

            gen_ids = model.backbone.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                eos_token_id=[END_ID, EOS, PAD],  # stop when </OBJS> or EOS or PAD
            )
            text = tokenizer.decode(gen_ids[0], skip_special_tokens=False)
            end = text.find("</OBJS>")
            if end != -1:
                text = text[:end + len("</OBJS>")]

            f.write(json.dumps({
                "problem": ex["problem"],
                "pred": text,
                "target": ex["target_text"],
            }, ensure_ascii=False) + "\n")

    print(f"Training complete. Predictions saved to {pred_path}")


if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Step,Training Loss
5,2.411400
10,0.666500
15,0.091400
20,0.049800
25,0.047100
30,0.042200
35,0.038800
40,0.041900
45,0.035000
50,0.038300


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


Training complete. Supervised JSON+regression model is trained on toy data.


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Training complete. Predictions saved to ./out/predictions.jsonl


In [ ]:
print("{\n  \"bodies\": [\n    {\n      \"id\": \"block1\",\n      \"type\": \"block\",\n      \"position\": {\n        \"x\": -0.009543716774490826,\n        \"y\": 0.015851960631336134\n      },\n      \"size\": {\n        \"width\": 0.6,\n        \"height\": 0.4\n      },\n      \"rotation_deg\": 30,\n      \"label\": \"\"\n    }\n  ],\n  \"surfaces\": [\n    {\n      \"id\": \"incline\",\n      \"type\": \"incline\",\n      \"angle_deg\": 30,\n      \"through\": {\n        \"x\": -0.4,\n        \"y\": -0.5\n      },\n      \"length\": 5.0\n    }\n  ],\n  \"forces\": [\n    {\n      \"id\": \"W\",\n      \"type\": \"weight\",\n      \"on\": \"block1\",\n      \"arrow\": {\n        \"start\": {\n          \"x\": 0.0,\n          \"y\": 0.0\n        },\n        \"angle_deg\": 270,\n        \"length\": 1.187057846185667\n      },\n      \"label\": {\n        \"text\": \"W\",\n        \"offset\": {\n          \"dx\": 0.0,\n          \"dy\": 0.0\n        }\n      }\n    },\n    {\n      \"id\": \"N\",\n      \"type\": \"normal\",\n      \"on\": \"block1\",\n      \"arrow\": {\n        \"start\": {\n          \"x\": 0.0,\n          \"y\": 0.0\n        },\n        \"angle_deg\": 120,\n        \"length\": 0.9\n      },\n      \"label\": {\n        \"text\": \"N\",\n        \"offset\": {\n          \"dx\": 0.0,\n          \"dy\": 0.0\n        }\n      }\n    },\n    {\n      \"id\": \"f\",\n      \"type\": \"friction\",\n      \"on\": \"block1\",\n      \"arrow\": {\n        \"start\": {\n          \"x\": 0.0,\n          \"y\": 0.0\n        },\n        \"angle_deg\": 30,\n        \"length\": 0.8\n      },\n      \"label\": {\n        \"text\": \"f\",\n        \"offset\": {\n          \"dx\": 0.0,\n          \"dy\": 0.0\n        }\n      }\n    }\n  ],\n  \"labels\": {\n    \"show_axes\": true,\n    \"origin\": {\n      \"x\": 1.0,\n      \"y\": -2.0\n    }\n  }\n}\n</OBJS>")

{
  "bodies": [
    {
      "id": "block1",
      "type": "block",
      "position": {
        "x": -0.009543716774490826,
        "y": 0.015851960631336134
      },
      "size": {
        "width": 0.6,
        "height": 0.4
      },
      "rotation_deg": 30,
      "label": ""
    }
  ],
  "surfaces": [
    {
      "id": "incline",
      "type": "incline",
      "angle_deg": 30,
      "through": {
        "x": -0.4,
        "y": -0.5
      },
      "length": 5.0
    }
  ],
  "forces": [
    {
      "id": "W",
      "type": "weight",
      "on": "block1",
      "arrow": {
        "start": {
          "x": 0.0,
          "y": 0.0
        },
        "angle_deg": 270,
        "length": 1.187057846185667
      },
      "label": {
        "text": "W",
        "offset": {
          "dx": 0.0,
          "dy": 0.0
        }
      }
    },
    {
      "id": "N",
      "type": "normal",
      "on": "block1",
      "arrow": {
        "start": {
          "x": 0.0,
          "y": 0.0
        },
  

# B

In [3]:
"""
End-to-end *supervised-only* overfit of a tiny JSON diagram generator.

Key fixes:
- Compact JSON targets (separators=(",", ":"))
- Longer max_len so prompt|target fits (1024 here)
- Same prefix for train & inference
- Deterministic decoding; stop at </OBJS>
- No jitter (exact overfit) by default
"""

import os, json, random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 0) Toy problems & gold JSONs
# ----------------------------
problems = [
    "A block of mass m rests on a 30° incline. Draw a free body diagram with W, N, and friction f.",
    "A block of mass m rests on a 45° incline. Draw a free body diagram with W, N, and friction f.",
    "A block of mass m rests on a 15° incline. Draw a free body diagram with W, N, and friction f."
]

jsons = [
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 30, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 30,
             "through": {"x": -0.4, "y": -0.5},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 120, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 30, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    },
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 45, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 45,
             "through": {"x": -0.4, "y": -0.7},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 135, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 45, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    },
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 15, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 15,
             "through": {"x": -0.4, "y": -0.3},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 105, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 15, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    }
]

#jsons = jsons[0:1]

# -----------------------------------------
# 1) JSON-aware <OBJ> injector (compact out)
# -----------------------------------------
def inject_obj_tokens_json(data_or_str) -> str:
    """Return compact JSON string with <OBJ> inserted before each list item; append </OBJS>."""
    if isinstance(data_or_str, str):
        data = json.loads(data_or_str)
    else:
        data = data_or_str

    def comp(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

    bodies   = data.get("bodies",   []); bodies   = bodies   if isinstance(bodies, list)   else []
    surfaces = data.get("surfaces", []); surfaces = surfaces if isinstance(surfaces, list) else []
    forces   = data.get("forces",   []); forces   = forces   if isinstance(forces, list)   else []
    labels   = data.get("labels",   None)

    parts = []
    parts.append("{")

    parts.append('"bodies":[')
    for i, o in enumerate(bodies):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"surfaces":[')
    for i, o in enumerate(surfaces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"forces":[')
    for i, o in enumerate(forces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    if labels is not None:
        parts.append(',"labels":'); parts.append(comp(labels))

    parts.append("}")
    parts.append("</OBJS>")
    return "".join(parts)

# -------------------------
# 2) Dataset (exact 2 items)
# -------------------------
class DiagramDataset(Dataset):
    def __init__(self, num_samples: int = 2, jitter: bool = False):
        self.samples = []
        for i in range(num_samples):
            idx = i % 2
            j = json.loads(json.dumps(jsons[idx]))
            if jitter:
                j["bodies"][0]["position"]["x"] += random.uniform(-0.02, 0.02)
                j["bodies"][0]["position"]["y"] += random.uniform(-0.02, 0.02)
                j["forces"][0]["arrow"]["length"] += random.uniform(-0.05, 0.05)
            self.samples.append({"problem": problems[idx], "json": j})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        problem = item["problem"]
        j = item["json"]
        # Compact + markerized target (starts with '{', ends with </OBJS>)
        target_with_markers = inject_obj_tokens_json(j)
        return {
            "problem": problem,
            "target_text": target_with_markers,
            # placeholders kept for future set-head losses
            "gold_types": torch.zeros(0, dtype=torch.long),
            "gold_regs":  torch.zeros(0, 8),
            "gold_masks": torch.zeros(0, 8, dtype=torch.long),
        }

# -----------------------------------------
# 3) Collator: prompt|target concat, no BOS
# -----------------------------------------
PREFIX = "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
SUFFIX = "\n\nJSON:\n"  # NOTE: target already includes the leading '{'

@dataclass
class PromptTargetCollator:
    tokenizer: PreTrainedTokenizerBase
    max_len: int = 1024  # long enough to keep all target tokens

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        prompts = [PREFIX + b["problem"] + SUFFIX for b in batch]
        tails   = ["\n" + b["target_text"] for b in batch]  # target starts with '{'

        enc_p = self.tokenizer(
            prompts, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )
        enc_t = self.tokenizer(
            tails, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )

        pad_id = self.tokenizer.pad_token_id
        obj_tok_id = self.tokenizer.convert_tokens_to_ids("<OBJ>")

        input_ids, attn_masks, labels, obj_positions = [], [], [], []

        B = len(batch)
        for i in range(B):
            p_ids, p_att = enc_p.input_ids[i], enc_p.attention_mask[i]
            t_ids, t_att = enc_t.input_ids[i], enc_t.attention_mask[i]

            ids = torch.cat([p_ids, t_ids], dim=0)
            att = torch.cat([p_att, t_att], dim=0)

            if ids.size(0) > self.max_len:
                ids = ids[: self.max_len]
                att = att[: self.max_len]

            plen = min(int(p_att.sum().item()), self.max_len)
            avail_t = self.max_len - plen
            tlen = min(int(t_att.sum().item()), max(0, avail_t))

            lab = torch.full_like(ids, -100)
            start, end = plen, plen + tlen
            if end > start:
                lab[start:end] = ids[start:end]

            # right-pad to max_len
            if ids.size(0) < self.max_len:
                pad_len = self.max_len - ids.size(0)
                ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=ids.dtype)], dim=0)
                att = torch.cat([att, torch.zeros(pad_len, dtype=att.dtype)], dim=0)
                lab = torch.cat([lab, torch.full((pad_len,), -100, dtype=lab.dtype)], dim=0)

            # collect <OBJ> positions inside target span (for future set-heads)
            obj_pos = []
            if end > start:
                rel = (ids[start:end] == obj_tok_id).nonzero(as_tuple=False).flatten()
                obj_pos = (rel + start).tolist()

            input_ids.append(ids)
            attn_masks.append(att.long())
            labels.append(lab)
            obj_positions.append(obj_pos)

        for i in range(len(batch)):
            plen = int((enc_p.attention_mask[i]).sum().item())
            tlen = int((enc_t.attention_mask[i]).sum().item())
            sup = int((labels[i] != -100).sum().item())
            # print(f"[collator sanity] sample {i}: prompt_len={plen}, target_len={tlen}, supervised={sup}")
            if sup == 0:
                raise ValueError("No supervised tokens! Increase max_len or fix label masking.")

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn_masks),
            "labels": torch.stack(labels),
            "obj_positions": obj_positions,
            "gold_types": [b["gold_types"] for b in batch],
            "gold_regs":  [b["gold_regs"]  for b in batch],
            "gold_masks": [b["gold_masks"] for b in batch],
        }

# ------------------------------------------
# 4) Model wrapper (text-only supervised CE)
# ------------------------------------------
class DiagramWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, **batch):
        out = self.backbone(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            output_hidden_states=False,
            return_dict=True,
        )
        return {"loss": out.loss}

# ------------------------------------------
# 5) Generation helpers (deterministic)
# ------------------------------------------
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    end = text.find("</OBJS>")
    if end != -1:
        text = text[:end]
    first, last = text.find("{"), text.rfind("}")
    if first == -1 or last == -1 or last <= first:
        return None
    raw = text[first:last+1].replace("<OBJ> ", "").replace("<OBJ>", "")
    try:
        return json.loads(raw)
    except Exception:
        return None

@torch.no_grad()
def gen_one(model, tokenizer, problem: str, seed_first_key: bool = False, max_new_tokens=1200) -> str:
    # SAME prefix as in training
    prompt = PREFIX + problem + SUFFIX
    if seed_first_key:
        # helps tiny sets: gives first key to constrain the structure
        prompt += '{"bodies":['
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)

    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    stop_ids = [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id]

    out = model.backbone.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_k=0,
        #no_repeat_ngram_size=4,
        #repetition_penalty=1.1,
        eos_token_id=stop_ids,  # list supported in recent HF
    )
    text = tokenizer.decode(out[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    cut = text.find("</OBJS>")
    if cut != -1:
        text = text[:cut + len("</OBJS>")]
    return text

@torch.no_grad()
def teacher_forced_token_acc(backbone, dl):
    backbone.eval()
    tot, correct = 0, 0
    for batch in dl:
        ids = batch["input_ids"].to(backbone.device)
        att = batch["attention_mask"].to(backbone.device)
        labels = batch["labels"].to(backbone.device)
        logits = backbone(input_ids=ids, attention_mask=att).logits
        pred = logits.argmax(-1)
        mask = labels != -100
        tot += mask.sum().item()
        correct += (pred[mask] == labels[mask]).sum().item()
    acc = correct / max(1, tot)
    print(f"[teacher-forced] token acc: {acc:.4f}")
    return acc

# -------------------
# 6) Train & evaluate
# -------------------
def main():
    random.seed(0); torch.manual_seed(0)

    backbone_name = "Qwen/Qwen2.5-Coder-7B-Instruct" #"TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # small and friendly

    tokenizer = AutoTokenizer.from_pretrained(backbone_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # add special markers (<OBJ>, </OBJS>) *before* loading model.resize
    added = tokenizer.add_special_tokens({"additional_special_tokens": ["<OBJ>", "</OBJS>"]})

    backbone = AutoModelForCausalLM.from_pretrained(
        backbone_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else None,
    )
    backbone.config.use_cache = False
    if added > 0:
        backbone.resize_token_embeddings(len(tokenizer))
    if getattr(backbone.config, "pad_token_id", None) is None:
        backbone.config.pad_token_id = tokenizer.pad_token_id

    model = DiagramWrapper(backbone)

    train_ds = DiagramDataset(num_samples=2, jitter=False)  # exact two items; no jitter to overfit
    collator = PromptTargetCollator(tokenizer=tokenizer, max_len=1024)

    # Check one sample passes through collator correctly
    tmp = [train_ds[0]]
    out = collator(tmp)
    print(f"Collator debug: input shape={out['input_ids'].shape}, "
          f"supervised={(out['labels'] != -100).sum().item()}")


    args = TrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        weight_decay=0.0,
        num_train_epochs=150,       # push to memorize
        logging_steps=5,
        save_strategy="no",
        report_to=[],
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    trainer.train()

    # Teacher-forced token accuracy on the train set
    dl = DataLoader(train_ds, batch_size=1, shuffle=False, collate_fn=collator)
    teacher_forced_token_acc(model.backbone, dl)

    # Generation on both prompts (try with and without seeding the first key)
    os.makedirs("./out", exist_ok=True)
    with open("./out/preds.jsonl", "w", encoding="utf-8") as f:
        for p in problems:
            text = gen_one(model, tokenizer, p, seed_first_key=False, max_new_tokens=1200)
            obj = extract_json(text)
            if obj is None:
                # tiny sets often benefit from a tad more structure
                text = gen_one(model, tokenizer, p, seed_first_key=True, max_new_tokens=1200)
                obj = extract_json(text)
            f.write(json.dumps({"problem": p, "pred_text": text, "parsed_ok": obj is not None}, ensure_ascii=False) + "\n")
    print("Wrote ./out/preds.jsonl")

if __name__ == "__main__":
    main()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Collator debug: input shape=torch.Size([1, 1024]), supervised=315


Step,Training Loss
5,3.043700
10,1.013700
15,0.568800
20,0.096600
25,0.038300
30,0.086300
35,0.003200
40,0.000400
45,0.000100
50,0.000000


[teacher-forced] token acc: 0.0032
Wrote ./out/preds.jsonl


In [4]:
print("{\"bodies\":[<OBJ> {\"id\":\"block1\",\"type\":\"block\",\"position\":{\"x\":0.0,\"y\":0.0},\"size\":{\"width\":0.6,\"height\":0.4},\"rotation_deg\":1.1,\"label\":\"\"}],\"surfaces\":[<OBJ> {\"id\":\"incline\",\"type\":\"incline\",\"angle_deg\":0.9,\"through\":{\"x\":-0.4,\"y\":-0.5},\"length\":5.0}],\"forces\":[<OBJ> {\"id\":\"W\",\"type\":\"weight\",\"on\":\"block1\",\"arrow\":{\"start\":{\"x\":0.0,\"y\":0.0},\"angle_deg\":0.8,\"length\":0.7},\"label\":{\"text\":\"W\",\"offset\":{\"dx\":0.0,\"dy\":0.0}}},<OBJ> {\"id\":\"N\",\"type\":\"normal\",\"on\":\"block1\",\"arrow\":{\"start\":{\"x\":0.0,\"y\":0.0},\"angle_deg\":0.8,\"length\":0.7},\"label\":{\"text\":\"N\",\"offset\":{\"dx\":0.0,\"dy\":0.0}}},<OBJ> {\"id\":\"f\",\"type\":\"friction\",\"on\":\"block1\",\"arrow\":{\"start\":{\"x\":0.0,\"y\":0.0},\"angle_deg\":0.8,\"length\":0.7},\"label\":{\"text\":\"f\",\"offset\":{\"dx\":0.0,\"dy\":0.0}}}],\"labels\":{\"show_axes\":true,\"origin\":{\"x\":1.0,\"y\":-2.0}}}")

{"bodies":[<OBJ> {"id":"block1","type":"block","position":{"x":0.0,"y":0.0},"size":{"width":0.6,"height":0.4},"rotation_deg":1.1,"label":""}],"surfaces":[<OBJ> {"id":"incline","type":"incline","angle_deg":0.9,"through":{"x":-0.4,"y":-0.5},"length":5.0}],"forces":[<OBJ> {"id":"W","type":"weight","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":0.8,"length":0.7},"label":{"text":"W","offset":{"dx":0.0,"dy":0.0}}},<OBJ> {"id":"N","type":"normal","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":0.8,"length":0.7},"label":{"text":"N","offset":{"dx":0.0,"dy":0.0}}},<OBJ> {"id":"f","type":"friction","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":0.8,"length":0.7},"label":{"text":"f","offset":{"dx":0.0,"dy":0.0}}}],"labels":{"show_axes":true,"origin":{"x":1.0,"y":-2.0}}}


# C

In [1]:
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

# ---------- helpers ----------
def _tok_to_dev(tokenizer, text, device):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    return {k: v.to(device) for k, v in enc.items()}

class StopOnAnyIds(StoppingCriteria):
    def __init__(self, stop_ids): self.stop_ids = set(stop_ids)
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1].item() in self.stop_ids

def _gen_det(backbone, tokenizer, prompt_text, stop_ids, max_new_tokens=200):
    """Deterministic generate: returns full decoded (prompt + completion)."""
    enc = _tok_to_dev(tokenizer, prompt_text, backbone.device)
    stop = StoppingCriteriaList([StopOnAnyIds(stop_ids)])
    out = backbone.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_k=0,
        no_repeat_ngram_size=4,
        repetition_penalty=1.1,
        stopping_criteria=stop,
        eos_token_id=list(stop_ids),
    )
    return tokenizer.decode(out[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)

def _tail_after(full_text, anchor):
    """Return substring of full_text that comes AFTER the last occurrence of anchor."""
    j = full_text.rfind(anchor)
    return "" if j == -1 else full_text[j+len(anchor):]

def _take_balanced_list(tail):
    """
    Given text that starts immediately after an opening '[' (we assume we've just
    emitted '[' in the seed), scan forward and return the shortest prefix that
    balances to depth==0. Returns (inner_content, found_close:boolean).
    """
    depth = 1
    out = []
    for ch in tail:
        if ch == '[':
            depth += 1
            out.append(ch)
        elif ch == ']':
            depth -= 1
            if depth == 0:
                return ("".join(out), True)
            out.append(ch)
        else:
            out.append(ch)
    return ("".join(out), False)

def _take_balanced_obj(tail):
    """Same as above but for an object starting right after '{'."""
    depth = 1
    out = []
    for ch in tail:
        if ch == '{':
            depth += 1
            out.append(ch)
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return ("".join(out), True)
            out.append(ch)
        else:
            out.append(ch)
    return ("".join(out), False)

# ---------- main segmented generator ----------
def generate_segmented(model, tokenizer, problem, max_per_section=220):
    """
    Build valid JSON by emitting four segments:
      bodies[], surfaces[], forces[], labels{}
    We never split or regex the model’s output; we parse balanced brackets on the fly.
    """
    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    PAD    = tokenizer.pad_token_id
    EOS    = tokenizer.eos_token_id
    STOP   = {END_ID, PAD, EOS}

    prefix = (
        "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\n"
        "Problem:\n" + problem + "\n\nJSON:\n"
    )

    # 1) bodies
    seed = prefix + '{"bodies":['
    full = _gen_det(model.backbone, tokenizer, seed, STOP, max_new_tokens=max_per_section)
    tail = _tail_after(full, seed)       # content after '['
    bodies_inner, ok = _take_balanced_list(tail)
    if not ok:
        bodies_inner = ""  # fall back to empty list
    json_out = '{"bodies":[' + bodies_inner + ']'

    # 2) surfaces
    seed = prefix + json_out + ',"surfaces":['
    full = _gen_det(model.backbone, tokenizer, seed, STOP, max_new_tokens=max_per_section)
    tail = _tail_after(full, seed)
    surfaces_inner, ok = _take_balanced_list(tail)
    if not ok:
        surfaces_inner = ""
    json_out += ',"surfaces":[' + surfaces_inner + ']'

    # 3) forces
    seed = prefix + json_out + ',"forces":['
    full = _gen_det(model.backbone, tokenizer, seed, STOP, max_new_tokens=max_per_section)
    tail = _tail_after(full, seed)
    forces_inner, ok = _take_balanced_list(tail)
    if not ok:
        forces_inner = ""
    json_out += ',"forces":[' + forces_inner + ']'

    # 4) labels (object)
    seed = prefix + json_out + ',"labels":{'
    full = _gen_det(model.backbone, tokenizer, seed, STOP, max_new_tokens=max_per_section)
    tail = _tail_after(full, seed)
    labels_inner, ok = _take_balanced_obj(tail)
    if not ok:
        labels_inner = '"show_axes":true,"origin":{"x":1.0,"y":-2.0}'  # safe default
    json_out += ',"labels":{' + labels_inner + '}'

    final = json_out + '}</OBJS>'

    # small cleanups: normalize <OBJ> spacing and strip any training prefix that got echoed
    final = final.replace("<OBJ>  ", "<OBJ> ").replace("<OBJ>   ", "<OBJ> ")
    return final


## C.1

In [1]:
from transformers import LogitsProcessor
import torch

def apply_fast_gen_settings(model):
    """
    Switch the (already-trained) model into fast inference mode.
    Keeps training-time losses untouched—only used at eval.
    """
    model.backbone.eval()
    # KV cache ON -> drastically speeds up generation
    model.backbone.config.use_cache = True
    # Gradient checkpointing slows generation; turn it off at eval
    if hasattr(model.backbone, "gradient_checkpointing_disable"):
        model.backbone.gradient_checkpointing_disable()
    torch.set_grad_enabled(False)


def estimate_target_tokens(tokenizer, target_text_with_markers: str) -> int:
    """
    Estimate how many new tokens we need to generate for one example.
    Use the *actual* compact JSON target from your dataset to set a sane cap.
    """
    return len(tokenizer(target_text_with_markers, add_special_tokens=False).input_ids)


class FastJsonGuard(LogitsProcessor):
    """
    Very cheap guard that:
    - Starts working only *after* the first '{' has been emitted
      (so we don't slow down early steps).
    - Keeps an append-only text cache to avoid re-decoding the full prefix
      each step (only decode the delta).
    - Uses a short sliding window (~64 chars) for quick rule checks.

    Rules (minimal, safe):
    - Prevents a second '{' before the first '}' appears (common runaway failure).
    - If '</OBJS>' already appears in the cache, strongly bias toward EOS by
      suppressing all other tokens. (Your eos_token_id also includes the
      '</OBJS>' token id, so this is a belt-and-suspenders).
    """
    def __init__(self, tokenizer, ob_end_token="</OBJS>", window_chars=64, penalty=50.0):
        self.tok = tokenizer
        self.cache_text = ""
        self.last_len = 0
        self.window_chars = window_chars
        self.penalty = penalty
        self.end_marker = ob_end_token

        # Precompute ids we might want to nudge/penalize
        self.open_brace_id = self._tok("{")
        self.close_brace_id = self._tok("}")
        # In some tokenizers, markers are single tokens, in others, multi-token pieces.
        # We only *detect* the text pattern and suppress *non-eos* after it's present.
        # (We don't force-emit end marker here; we only allow EOS finishing.)
        # Nothing else to precompute.

    def _tok(self, s: str):
        # Best-effort: if it maps to multiple tokens, we won't use it for single-id suppression.
        ids = self.tok.encode(s, add_special_tokens=False)
        return ids[0] if len(ids) == 1 else None

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        # Greedy/one-sample generation path: shape [1, seq]
        cur = input_ids[0].tolist()
        if len(cur) > self.last_len:
            delta = self.tok.decode(cur[self.last_len:], skip_special_tokens=False)
            self.cache_text += delta
            self.last_len = len(cur)

        # Work on a short window
        window = self.cache_text[-self.window_chars:]

        # If '</OBJS>' already present, suppress any further non-EOS continuation
        if self.end_marker in self.cache_text:
            # Heavily penalize all tokens except EOS set (handled by eos_token_id)
            # We can't edit eos list here, but we can strongly suppress everything
            # and let the eos_token_id list terminate immediately.
            scores -= self.penalty
            return scores

        # Simple brace sanity: after we've seen the first '{', disallow *another*
        # '{' before any '}' occurs (this reduces malformed duplication cascades).
        if "{" in self.cache_text and "}" not in self.cache_text and self.open_brace_id is not None:
            scores[0, self.open_brace_id] -= self.penalty

        return scores


@torch.no_grad()
def gen_one_fast(
    model, tokenizer, problem: str, *,
    end_token="</OBJS>",
    seed_first_key=False,
    max_new_tokens=1024,
    add_guard=True,
    margin=32
) -> str:
    """
    Deterministic, fast generation for one prompt.
    - Re-enables KV cache
    - Uses greedy decoding with early stopping on '</OBJS>' | EOS | PAD
    - (Optional) seeds '{"bodies":[' to stabilize tiny overfits
    - (Optional) cheap constrained logits processor (FastJsonGuard)
    - Caps max_new_tokens using a length estimate + small margin
    """
    # HOOK 1: ensure fast gen settings are applied
    apply_fast_gen_settings(model)

    # Build the exact same prefix used during training
    PREFIX = "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
    SUFFIX = "\n\nJSON:\n"
    prompt = PREFIX + problem + SUFFIX
    if seed_first_key:
        prompt += '{"bodies":['  # lightweight “constrained start”

    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)

    # Build eos set
    END_ID = tokenizer.convert_tokens_to_ids(end_token)
    eos_ids = [tok for tok in [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id] if tok is not None]

    # Estimate a realistic cap (keeps decoding snappy)
    # If you have the gold target handy, use that; otherwise fall back to a safe cap.
    # Here we try a best-effort with a synthetic minimal body:
    rough_target = '{"bodies":[<OBJ> {"id":"block1","type":"block","position":{"x":0,"y":0},"size":{"width":0.6,"height":0.4},"rotation_deg":30,"label":""}],"surfaces":[<OBJ> {"id":"incline","type":"incline","angle_deg":30,"through":{"x":-0.4,"y":-0.5},"length":5.0}],"forces":[<OBJ> {"id":"W","type":"weight","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":270,"length":1.1},"label":{"text":"W","offset":{"dx":0.0,"dy":0.0}}},<OBJ> {"id":"N","type":"normal","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":120,"length":0.9},"label":{"text":"N","offset":{"dx":0.0,"dy":0.0}}},<OBJ> {"id":"f","type":"friction","on":"block1","arrow":{"start":{"x":0.0,"y":0.0},"angle_deg":30,"length":0.8},"label":{"text":"f","offset":{"dx":0.0,"dy":0.0}}}],"labels":{"show_axes":true,"origin":{"x":1.0,"y":-2.0}}}</OBJS>'
    est_len = estimate_target_tokens(tokenizer, rough_target)
    max_new = min(max_new_tokens, est_len + margin)

    # Optional guard
    processors = []
    if add_guard:
        processors.append(FastJsonGuard(tokenizer))

    out_ids = model.backbone.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,                # deterministic
        temperature=0.0,
        top_k=0,
        num_beams=1,
        no_repeat_ngram_size=4,
        repetition_penalty=1.10,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.eos_token_id,
        logits_processor=processors if processors else None,
        early_stopping=True,
    )

    text = tokenizer.decode(out_ids[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    cut = text.find(end_token)
    if cut != -1:
        text = text[:cut + len(end_token)]
    return text

## C.2

In [3]:
@torch.no_grad()
def gen_one_constrained(model, tokenizer, problem: str, max_new_tokens=512) -> str:
    # Same training prefix
    PREFIX = "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
    SUFFIX = "\n\nJSON:\n"
    prompt = PREFIX + problem + SUFFIX
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)

    # Build constraint
    prefix_fn = build_schema_template(tokenizer)

    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    eos_ids = [tok for tok in [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id] if tok is not None]

    out = model.backbone.generate(
        **enc,
        do_sample=False,
        temperature=0.0,
        top_k=0,
        num_beams=1,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.eos_token_id,
        prefix_allowed_tokens_fn=prefix_fn,  # <--- HARD constraint
        use_cache=True,
        no_repeat_ngram_size=0,              # not needed with hard template
        repetition_penalty=1.0,
        early_stopping=True,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    cut = text.find("</OBJS>")
    if cut != -1:
        text = text[:cut + len("</OBJS>")]
    return text


## C.3

In [2]:
"""
End-to-end *supervised-only* overfit of a tiny JSON diagram generator.

Key fixes:
- Compact JSON targets (separators=(",", ":"))
- Longer max_len so prompt|target fits (1024 here)
- Same prefix for train & inference
- Deterministic decoding; stop at </OBJS>
- No jitter (exact overfit) by default
"""

import os, json, random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 0) Toy problems & gold JSONs
# ----------------------------
problems = [
    "A block of mass m rests on a 30° incline. Draw a free body diagram with W, N, and friction f.",
    "A block of mass m rests on a 45° incline. Draw a free body diagram with W, N, and friction f.",
]

jsons = [
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 30, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 30,
             "through": {"x": -0.4, "y": -0.5},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 120, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 30, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    },
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 45, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 45,
             "through": {"x": -0.4, "y": -0.7},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 135, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 45, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    }
]

jsons = jsons[0:1]

# -----------------------------------------
# 1) JSON-aware <OBJ> injector (compact out)
# -----------------------------------------
def inject_obj_tokens_json(data_or_str) -> str:
    """Return compact JSON string with <OBJ> inserted before each list item; append </OBJS>."""
    if isinstance(data_or_str, str):
        data = json.loads(data_or_str)
    else:
        data = data_or_str

    def comp(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

    bodies   = data.get("bodies",   []); bodies   = bodies   if isinstance(bodies, list)   else []
    surfaces = data.get("surfaces", []); surfaces = surfaces if isinstance(surfaces, list) else []
    forces   = data.get("forces",   []); forces   = forces   if isinstance(forces, list)   else []
    labels   = data.get("labels",   None)

    parts = []
    parts.append("{")

    parts.append('"bodies":[')
    for i, o in enumerate(bodies):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"surfaces":[')
    for i, o in enumerate(surfaces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"forces":[')
    for i, o in enumerate(forces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    if labels is not None:
        parts.append(',"labels":'); parts.append(comp(labels))

    parts.append("}")
    parts.append("</OBJS>")
    return "".join(parts)

# -------------------------
# 2) Dataset (exact 2 items)
# -------------------------
class DiagramDataset(Dataset):
    def __init__(self, num_samples: int = 2, jitter: bool = False):
        self.samples = []
        for i in range(num_samples):
            idx = i % 2
            j = json.loads(json.dumps(jsons[idx]))
            if jitter:
                j["bodies"][0]["position"]["x"] += random.uniform(-0.02, 0.02)
                j["bodies"][0]["position"]["y"] += random.uniform(-0.02, 0.02)
                j["forces"][0]["arrow"]["length"] += random.uniform(-0.05, 0.05)
            self.samples.append({"problem": problems[idx], "json": j})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        problem = item["problem"]
        j = item["json"]
        # Compact + markerized target (starts with '{', ends with </OBJS>)
        target_with_markers = inject_obj_tokens_json(j)
        return {
            "problem": problem,
            "target_text": target_with_markers,
            # placeholders kept for future set-head losses
            "gold_types": torch.zeros(0, dtype=torch.long),
            "gold_regs":  torch.zeros(0, 8),
            "gold_masks": torch.zeros(0, 8, dtype=torch.long),
        }

# -----------------------------------------
# 3) Collator: prompt|target concat, no BOS
# -----------------------------------------
PREFIX = "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
SUFFIX = "\n\nJSON:\n"  # NOTE: target already includes the leading '{'

@dataclass
class PromptTargetCollator:
    tokenizer: PreTrainedTokenizerBase
    max_len: int = 1024  # long enough to keep all target tokens

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        prompts = [PREFIX + b["problem"] + SUFFIX for b in batch]
        tails   = ["\n" + b["target_text"] for b in batch]  # target starts with '{'

        enc_p = self.tokenizer(
            prompts, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )
        enc_t = self.tokenizer(
            tails, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )

        pad_id = self.tokenizer.pad_token_id
        obj_tok_id = self.tokenizer.convert_tokens_to_ids("<OBJ>")

        input_ids, attn_masks, labels, obj_positions = [], [], [], []

        B = len(batch)
        for i in range(B):
            p_ids, p_att = enc_p.input_ids[i], enc_p.attention_mask[i]
            t_ids, t_att = enc_t.input_ids[i], enc_t.attention_mask[i]

            ids = torch.cat([p_ids, t_ids], dim=0)
            att = torch.cat([p_att, t_att], dim=0)

            if ids.size(0) > self.max_len:
                ids = ids[: self.max_len]
                att = att[: self.max_len]

            plen = min(int(p_att.sum().item()), self.max_len)
            avail_t = self.max_len - plen
            tlen = min(int(t_att.sum().item()), max(0, avail_t))

            lab = torch.full_like(ids, -100)
            start, end = plen, plen + tlen
            if end > start:
                lab[start:end] = ids[start:end]

            # right-pad to max_len
            if ids.size(0) < self.max_len:
                pad_len = self.max_len - ids.size(0)
                ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=ids.dtype)], dim=0)
                att = torch.cat([att, torch.zeros(pad_len, dtype=att.dtype)], dim=0)
                lab = torch.cat([lab, torch.full((pad_len,), -100, dtype=lab.dtype)], dim=0)

            # collect <OBJ> positions inside target span (for future set-heads)
            obj_pos = []
            if end > start:
                rel = (ids[start:end] == obj_tok_id).nonzero(as_tuple=False).flatten()
                obj_pos = (rel + start).tolist()

            input_ids.append(ids)
            attn_masks.append(att.long())
            labels.append(lab)
            obj_positions.append(obj_pos)

        for i in range(len(batch)):
            plen = int((enc_p.attention_mask[i]).sum().item())
            tlen = int((enc_t.attention_mask[i]).sum().item())
            sup = int((labels[i] != -100).sum().item())
            # print(f"[collator sanity] sample {i}: prompt_len={plen}, target_len={tlen}, supervised={sup}")
            if sup == 0:
                raise ValueError("No supervised tokens! Increase max_len or fix label masking.")

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn_masks),
            "labels": torch.stack(labels),
            "obj_positions": obj_positions,
            "gold_types": [b["gold_types"] for b in batch],
            "gold_regs":  [b["gold_regs"]  for b in batch],
            "gold_masks": [b["gold_masks"] for b in batch],
        }

# ------------------------------------------
# 4) Model wrapper (text-only supervised CE)
# ------------------------------------------
class DiagramWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, **batch):
        out = self.backbone(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            output_hidden_states=False,
            return_dict=True,
        )
        return {"loss": out.loss}

# ------------------------------------------
# 5) Generation helpers (deterministic)
# ------------------------------------------
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    end = text.find("</OBJS>")
    if end != -1:
        text = text[:end]
    first, last = text.find("{"), text.rfind("}")
    if first == -1 or last == -1 or last <= first:
        return None
    raw = text[first:last+1].replace("<OBJ> ", "").replace("<OBJ>", "")
    try:
        return json.loads(raw)
    except Exception:
        return None

@torch.no_grad()
def gen_one(model, tokenizer, problem: str, seed_first_key: bool = False, max_new_tokens=1200) -> str:
    prompt = PREFIX + problem + SUFFIX
    if seed_first_key:
        prompt += '{"bodies":['
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)

    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    stop_ids = [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id]

    logits_processor = ConstrainedJsonLogitsProcessor(tokenizer)

    out = model.backbone.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_k=0,
        no_repeat_ngram_size=4,
        repetition_penalty=1.1,
        eos_token_id=stop_ids,                 # <- still stop at </OBJS>
        logits_processor=[logits_processor],   # <- NEW: constrained decoding
    )
    text = tokenizer.decode(out[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    cut = text.find("</OBJS>")
    if cut != -1:
        text = text[:cut + len("</OBJS>")]
    return text


@torch.no_grad()
def teacher_forced_token_acc(backbone, dl):
    backbone.eval()
    tot, correct = 0, 0
    for batch in dl:
        ids = batch["input_ids"].to(backbone.device)
        att = batch["attention_mask"].to(backbone.device)
        labels = batch["labels"].to(backbone.device)
        logits = backbone(input_ids=ids, attention_mask=att).logits
        pred = logits.argmax(-1)
        mask = labels != -100
        tot += mask.sum().item()
        correct += (pred[mask] == labels[mask]).sum().item()
    acc = correct / max(1, tot)
    print(f"[teacher-forced] token acc: {acc:.4f}")
    return acc

# -------------------
# 6) Train & evaluate
# -------------------
def main():
    random.seed(0); torch.manual_seed(0)

    backbone_name = "Qwen/Qwen2.5-Coder-7B-Instruct" #"TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # small and friendly

    tokenizer = AutoTokenizer.from_pretrained(backbone_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # add special markers (<OBJ>, </OBJS>) *before* loading model.resize
    added = tokenizer.add_special_tokens({"additional_special_tokens": ["<OBJ>", "</OBJS>"]})

    backbone = AutoModelForCausalLM.from_pretrained(
        backbone_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else None,
    )
    backbone.config.use_cache = False
    if added > 0:
        backbone.resize_token_embeddings(len(tokenizer))
    if getattr(backbone.config, "pad_token_id", None) is None:
        backbone.config.pad_token_id = tokenizer.pad_token_id

    model = DiagramWrapper(backbone)

    train_ds = DiagramDataset(num_samples=1, jitter=False)  # exact two items; no jitter to overfit
    collator = PromptTargetCollator(tokenizer=tokenizer, max_len=1024)

    # Check one sample passes through collator correctly
    tmp = [train_ds[0]]
    out = collator(tmp)
    print(f"Collator debug: input shape={out['input_ids'].shape}, "
          f"supervised={(out['labels'] != -100).sum().item()}")


    args = TrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        weight_decay=0.0,
        num_train_epochs=100,       # push to memorize
        logging_steps=5,
        save_strategy="no",
        report_to=[],
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    trainer.train()

    # Teacher-forced token accuracy on the train set
    dl = DataLoader(train_ds, batch_size=1, shuffle=False, collate_fn=collator)
    teacher_forced_token_acc(model.backbone, dl)

    # Generation on both prompts (try with and without seeding the first key)
    os.makedirs("./out", exist_ok=True)
    with open("./out/preds.jsonl", "w", encoding="utf-8") as f:
        for p in problems:
            text = generate_segmented(model, tokenizer, p, max_per_section=200)
            #text = gen_one(model, tokenizer, PREFIX + p + SUFFIX, seed_first_key=False, max_new_tokens=1200)
            obj = extract_json(text)
            """
            if obj is None and False:
                # tiny sets often benefit from a tad more structure
                text = gen_one_constrained(model, tokenizer, p, seed_first_key=True, max_new_tokens=1200)
                obj = extract_json(text)
            """
            f.write(json.dumps({"problem": p, "pred_text": text, "parsed_ok": obj is not None}, ensure_ascii=False) + "\n")
    print("Wrote ./out/preds.jsonl")

if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Collator debug: input shape=torch.Size([1, 1024]), supervised=315


Step,Training Loss
5,2.965000
10,1.728600
15,0.325200
20,0.311800
25,0.078400
30,0.178100
35,0.021500
40,0.005900
45,0.000800
50,0.000100


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[teacher-forced] token acc: 0.0000
Wrote ./out/preds.jsonl


In [3]:
print("{\"bodies\":[<OBJ> {\"id\":\"block1\",\"type\":\"block\",\"position\":{\"x\":0.0,\"y\":0.5},\"size\":{\"width\":0.6,\"height\":0.4},\"rotation_deg\":30,\"label\":\"\"}],\"surfaces\":[<OBJ> {\"incline\",\"incline\",\"angle_deg\":3270,\"through\":{\"x\":-0.4,\"y\":-0.5],\"forces\":[],\"labels\":{show_axes\":true,\"origin\":{\"x\":1.0,\"dy\":0.9}}}")

{"bodies":[<OBJ> {"id":"block1","type":"block","position":{"x":0.0,"y":0.5},"size":{"width":0.6,"height":0.4},"rotation_deg":30,"label":""}],"surfaces":[<OBJ> {"incline","incline","angle_deg":3270,"through":{"x":-0.4,"y":-0.5],"forces":[],"labels":{show_axes":true,"origin":{"x":1.0,"dy":0.9}}}


# D

In [3]:
!pip install lm-format-enforcer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00


In [5]:
# --- lm-format-enforcer (version-adaptive import) ---
from lmformatenforcer import JsonSchemaParser
try:
    # newer naming
    from lmformatenforcer.integrations.transformers import (
        build_token_enforcer_tokenizer_data,
        EnforcerLogitsProcessor as _EnforcerLogitsProcessor,
    )
except ImportError:
    # older naming
    from lmformatenforcer.integrations.transformers import (
        build_token_enforcer_tokenizer_data,
        TokenEnforcerLogitsProcessor as _EnforcerLogitsProcessor,
    )

from transformers import LogitsProcessorList
# 1) Define your minimal JSON schema (start strict, then expand):
schema = {
    "type": "object",
    "properties": {
        "bodies":   {"type": "array"},
        "surfaces": {"type": "array"},
        "forces":   {"type": "array"},
        "labels":   {"type": "object"},
    },
    "required": ["bodies", "surfaces", "forces", "labels"],
    "additionalProperties": True  # allow extra keys while you iterate
}

# 2) Build parser + tokenizer data + logits processor
parser = JsonSchemaParser(schema)
tok_data = build_token_enforcer_tokenizer_data(tokenizer)
lp = LogitsProcessorList([_EnforcerLogitsProcessor(parser, tok_data)])

# 3) Usual stop conditions (include your custom end token):
END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
stop_ids = [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id]

# 4) Generate with constraints
enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)
out_ids = model.backbone.generate(
    **enc,
    max_new_tokens=512,
    do_sample=False,
    logits_processor=lp,
    eos_token_id=stop_ids,
)
text = tokenizer.decode(out_ids[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)


ImportError: cannot import name 'TokenEnforcerLogitsProcessor' from 'lmformatenforcer.integrations.transformers' (/usr/local/lib/python3.12/dist-packages/lmformatenforcer/integrations/transformers.py)

In [2]:
"""
End-to-end *supervised-only* overfit of a tiny JSON diagram generator.

Key fixes:
- Compact JSON targets (separators=(",", ":"))
- Longer max_len so prompt|target fits (1024 here)
- Same prefix for train & inference
- Deterministic decoding; stop at </OBJS>
- No jitter (exact overfit) by default
"""

import os, json, random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 0) Toy problems & gold JSONs
# ----------------------------
problems = [
    "A block of mass m rests on a 30° incline. Draw a free body diagram with W, N, and friction f.",
    "A block of mass m rests on a 45° incline. Draw a free body diagram with W, N, and friction f.",
]

jsons = [
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 30, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 30,
             "through": {"x": -0.4, "y": -0.5},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 120, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 30, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    },
    {
        "bodies": [
            {"id": "block1", "type": "block",
             "position": {"x": 0.0, "y": 0.0},
             "size": {"width": 0.6, "height": 0.4},
             "rotation_deg": 45, "label": ""}
        ],
        "surfaces": [
            {"id": "incline", "type": "incline",
             "angle_deg": 45,
             "through": {"x": -0.4, "y": -0.7},
             "length": 5.0}
        ],
        "forces": [
            {"id": "W", "type": "weight", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 270, "length": 1.1},
             "label": {"text": "W", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "N", "type": "normal", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 135, "length": 0.9},
             "label": {"text": "N", "offset": {"dx": 0.0, "dy": 0.0}}},
            {"id": "f", "type": "friction", "on": "block1",
             "arrow": {"start": {"x": 0.0, "y": 0.0}, "angle_deg": 45, "length": 0.8},
             "label": {"text": "f", "offset": {"dx": 0.0, "dy": 0.0}}}
        ],
        "labels": {"show_axes": True, "origin": {"x": 1.0, "y": -2.0}}
    }
]

jsons = jsons[0:1]

# -----------------------------------------
# 1) JSON-aware <OBJ> injector (compact out)
# -----------------------------------------
def inject_obj_tokens_json(data_or_str) -> str:
    """Return compact JSON string with <OBJ> inserted before each list item; append </OBJS>."""
    if isinstance(data_or_str, str):
        data = json.loads(data_or_str)
    else:
        data = data_or_str

    def comp(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

    bodies   = data.get("bodies",   []); bodies   = bodies   if isinstance(bodies, list)   else []
    surfaces = data.get("surfaces", []); surfaces = surfaces if isinstance(surfaces, list) else []
    forces   = data.get("forces",   []); forces   = forces   if isinstance(forces, list)   else []
    labels   = data.get("labels",   None)

    parts = []
    parts.append("{")

    parts.append('"bodies":[')
    for i, o in enumerate(bodies):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"surfaces":[')
    for i, o in enumerate(surfaces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    parts.append(',"forces":[')
    for i, o in enumerate(forces):
        if i: parts.append(",")
        parts.append("<OBJ> "); parts.append(comp(o))
    parts.append("]")

    if labels is not None:
        parts.append(',"labels":'); parts.append(comp(labels))

    parts.append("}")
    parts.append("</OBJS>")
    return "".join(parts)

# -------------------------
# 2) Dataset (exact 2 items)
# -------------------------
class DiagramDataset(Dataset):
    def __init__(self, num_samples: int = 2, jitter: bool = False):
        self.samples = []
        for i in range(num_samples):
            idx = i % 2
            j = json.loads(json.dumps(jsons[idx]))
            if jitter:
                j["bodies"][0]["position"]["x"] += random.uniform(-0.02, 0.02)
                j["bodies"][0]["position"]["y"] += random.uniform(-0.02, 0.02)
                j["forces"][0]["arrow"]["length"] += random.uniform(-0.05, 0.05)
            self.samples.append({"problem": problems[idx], "json": j})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        problem = item["problem"]
        j = item["json"]
        # Compact + markerized target (starts with '{', ends with </OBJS>)
        target_with_markers = inject_obj_tokens_json(j)
        return {
            "problem": problem,
            "target_text": target_with_markers,
            # placeholders kept for future set-head losses
            "gold_types": torch.zeros(0, dtype=torch.long),
            "gold_regs":  torch.zeros(0, 8),
            "gold_masks": torch.zeros(0, 8, dtype=torch.long),
        }

# -----------------------------------------
# 3) Collator: prompt|target concat, no BOS
# -----------------------------------------
PREFIX = "You are a diagram JSON generator. Output ONLY JSON matching the schema.\n\nProblem:\n"
SUFFIX = "\n\nJSON:\n"  # NOTE: target already includes the leading '{'

@dataclass
class PromptTargetCollator:
    tokenizer: PreTrainedTokenizerBase
    max_len: int = 1024  # long enough to keep all target tokens

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        prompts = [PREFIX + b["problem"] + SUFFIX for b in batch]
        tails   = ["\n" + b["target_text"] for b in batch]  # target starts with '{'

        enc_p = self.tokenizer(
            prompts, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )
        enc_t = self.tokenizer(
            tails, padding=False, truncation=True, max_length=self.max_len,
            return_tensors="pt", add_special_tokens=False
        )

        pad_id = self.tokenizer.pad_token_id
        obj_tok_id = self.tokenizer.convert_tokens_to_ids("<OBJ>")

        input_ids, attn_masks, labels, obj_positions = [], [], [], []

        B = len(batch)
        for i in range(B):
            p_ids, p_att = enc_p.input_ids[i], enc_p.attention_mask[i]
            t_ids, t_att = enc_t.input_ids[i], enc_t.attention_mask[i]

            ids = torch.cat([p_ids, t_ids], dim=0)
            att = torch.cat([p_att, t_att], dim=0)

            if ids.size(0) > self.max_len:
                ids = ids[: self.max_len]
                att = att[: self.max_len]

            plen = min(int(p_att.sum().item()), self.max_len)
            avail_t = self.max_len - plen
            tlen = min(int(t_att.sum().item()), max(0, avail_t))

            lab = torch.full_like(ids, -100)
            start, end = plen, plen + tlen
            if end > start:
                lab[start:end] = ids[start:end]

            # right-pad to max_len
            if ids.size(0) < self.max_len:
                pad_len = self.max_len - ids.size(0)
                ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=ids.dtype)], dim=0)
                att = torch.cat([att, torch.zeros(pad_len, dtype=att.dtype)], dim=0)
                lab = torch.cat([lab, torch.full((pad_len,), -100, dtype=lab.dtype)], dim=0)

            # collect <OBJ> positions inside target span (for future set-heads)
            obj_pos = []
            if end > start:
                rel = (ids[start:end] == obj_tok_id).nonzero(as_tuple=False).flatten()
                obj_pos = (rel + start).tolist()

            input_ids.append(ids)
            attn_masks.append(att.long())
            labels.append(lab)
            obj_positions.append(obj_pos)

        for i in range(len(batch)):
            plen = int((enc_p.attention_mask[i]).sum().item())
            tlen = int((enc_t.attention_mask[i]).sum().item())
            sup = int((labels[i] != -100).sum().item())
            # print(f"[collator sanity] sample {i}: prompt_len={plen}, target_len={tlen}, supervised={sup}")
            if sup == 0:
                raise ValueError("No supervised tokens! Increase max_len or fix label masking.")

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn_masks),
            "labels": torch.stack(labels),
            "obj_positions": obj_positions,
            "gold_types": [b["gold_types"] for b in batch],
            "gold_regs":  [b["gold_regs"]  for b in batch],
            "gold_masks": [b["gold_masks"] for b in batch],
        }

# ------------------------------------------
# 4) Model wrapper (text-only supervised CE)
# ------------------------------------------
class DiagramWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, **batch):
        out = self.backbone(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            output_hidden_states=False,
            return_dict=True,
        )
        return {"loss": out.loss}

# ------------------------------------------
# 5) Generation helpers (deterministic)
# ------------------------------------------
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    end = text.find("</OBJS>")
    if end != -1:
        text = text[:end]
    first, last = text.find("{"), text.rfind("}")
    if first == -1 or last == -1 or last <= first:
        return None
    raw = text[first:last+1].replace("<OBJ> ", "").replace("<OBJ>", "")
    try:
        return json.loads(raw)
    except Exception:
        return None

@torch.no_grad()
def gen_one(model, tokenizer, problem: str, seed_first_key: bool = False, max_new_tokens=1200) -> str:
    prompt = PREFIX + problem + SUFFIX
    if seed_first_key:
        prompt += '{"bodies":['
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.backbone.device)

    END_ID = tokenizer.convert_tokens_to_ids("</OBJS>")
    stop_ids = [END_ID, tokenizer.eos_token_id, tokenizer.pad_token_id]

    logits_processor = ConstrainedJsonLogitsProcessor(tokenizer)

    out = model.backbone.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_k=0,
        no_repeat_ngram_size=4,
        repetition_penalty=1.1,
        eos_token_id=stop_ids,                 # <- still stop at </OBJS>
        logits_processor=[logits_processor],   # <- NEW: constrained decoding
    )
    text = tokenizer.decode(out[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    cut = text.find("</OBJS>")
    if cut != -1:
        text = text[:cut + len("</OBJS>")]
    return text


@torch.no_grad()
def teacher_forced_token_acc(backbone, dl):
    backbone.eval()
    tot, correct = 0, 0
    for batch in dl:
        ids = batch["input_ids"].to(backbone.device)
        att = batch["attention_mask"].to(backbone.device)
        labels = batch["labels"].to(backbone.device)
        logits = backbone(input_ids=ids, attention_mask=att).logits
        pred = logits.argmax(-1)
        mask = labels != -100
        tot += mask.sum().item()
        correct += (pred[mask] == labels[mask]).sum().item()
    acc = correct / max(1, tot)
    print(f"[teacher-forced] token acc: {acc:.4f}")
    return acc

# -------------------
# 6) Train & evaluate
# -------------------
def main():
    random.seed(0); torch.manual_seed(0)

    backbone_name = "Qwen/Qwen2.5-Coder-7B-Instruct" #"TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # small and friendly

    tokenizer = AutoTokenizer.from_pretrained(backbone_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # add special markers (<OBJ>, </OBJS>) *before* loading model.resize
    added = tokenizer.add_special_tokens({"additional_special_tokens": ["<OBJ>", "</OBJS>"]})

    backbone = AutoModelForCausalLM.from_pretrained(
        backbone_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else None,
    )
    backbone.config.use_cache = False
    if added > 0:
        backbone.resize_token_embeddings(len(tokenizer))
    if getattr(backbone.config, "pad_token_id", None) is None:
        backbone.config.pad_token_id = tokenizer.pad_token_id

    model = DiagramWrapper(backbone)

    train_ds = DiagramDataset(num_samples=1, jitter=False)  # exact two items; no jitter to overfit
    collator = PromptTargetCollator(tokenizer=tokenizer, max_len=1024)

    # Check one sample passes through collator correctly
    tmp = [train_ds[0]]
    out = collator(tmp)
    print(f"Collator debug: input shape={out['input_ids'].shape}, "
          f"supervised={(out['labels'] != -100).sum().item()}")


    args = TrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        weight_decay=0.0,
        num_train_epochs=100,       # push to memorize
        logging_steps=5,
        save_strategy="no",
        report_to=[],
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    trainer.train()

    # Teacher-forced token accuracy on the train set
    dl = DataLoader(train_ds, batch_size=1, shuffle=False, collate_fn=collator)
    teacher_forced_token_acc(model.backbone, dl)

    # Generation on both prompts (try with and without seeding the first key)
    os.makedirs("./out", exist_ok=True)
    with open("./out/preds.jsonl", "w", encoding="utf-8") as f:
        for p in problems:
            text = generate_jsonformer(model, tokenizer, p, insert_markers=True)
            #text = gen_one(model, tokenizer, PREFIX + p + SUFFIX, seed_first_key=False, max_new_tokens=1200)
            obj = extract_json(text)
            """
            if obj is None and False:
                # tiny sets often benefit from a tad more structure
                text = gen_one_constrained(model, tokenizer, p, seed_first_key=True, max_new_tokens=1200)
                obj = extract_json(text)
            """
            f.write(json.dumps({"problem": p, "pred_text": text, "parsed_ok": obj is not None}, ensure_ascii=False) + "\n")
    print("Wrote ./out/preds.jsonl")

if __name__ == "__main__":
    main()


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)